In [1]:
import logging
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")
sys.path

['research',
 'Singer',
 '/home/kirill/code/hackatons/XLabs-Hack-2024',
 '/home/kirill/.pyenv/versions/3.9.20/lib/python39.zip',
 '/home/kirill/.pyenv/versions/3.9.20/lib/python3.9',
 '/home/kirill/.pyenv/versions/3.9.20/lib/python3.9/lib-dynload',
 '',
 '/home/kirill/.cache/pypoetry/virtualenvs/xlabs-hack-2024-zh4tEKva-py3.9/lib/python3.9/site-packages']

In [4]:
!curl 2ip.ru

185.119.1.93


In [ ]:
from Singer.fairseq.checkpoint_utils import load_model_ensemble_and_task_from_hf_hub

models, cfg, task = load_model_ensemble_and_task_from_hf_hub(
    "Cyanbox/Prompt-Singer"
)

print(models)

In [6]:
from research.PromptSinger.dataset.tokenizer.soundstream.AudioTokenizer import AudioTokenizer
import torch

# Load the audio tokenizer with a checkpoint path
def load_audio_tokenizer(ckpt_path='/home/kirill/.cache/fairseq/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/codec/ckpt_01135000.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    audio_tokenizer = AudioTokenizer(ckpt_path=ckpt_path, device=device)
    return audio_tokenizer

audio_tokenizer = load_audio_tokenizer()

/home/kirill/.cache/pypoetry/virtualenvs/xlabs-hack-2024-zh4tEKva-py3.9/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/kirill/code/hackatons/XLabs-Hack-2024/Singer/research/PromptSinger/dataset/tokenizer/soundstream/AudioTokenizer.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are

In [11]:
import soundfile as sf

def prepare_prompt(lyrics, voice_description):
    return f"Singing: {lyrics}\nVoice style: {voice_description}"

def generate_vocal_audio(model, tokenizer, audio_tokenizer, prompt, output_file="generated_vocal.wav"):
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=500)

    # Decode generated tokens to audio tokens
    generated_tokens = outputs[0]
    audio_tokens = audio_tokenizer.decode(generated_tokens)

    # Convert audio tokens to waveform
    waveform = audio_tokenizer.tokens_to_waveform(audio_tokens)
    
    # Save the waveform as a WAV file
    sf.write(output_file, waveform.cpu().numpy(), audio_tokenizer.sample_rate)
    print(f"Vocal audio saved to {output_file}")

In [12]:
lyrics = "Twinkle, twinkle, little star, how I wonder what you are."
voice_description = "Soft and gentle female voice, with a dreamy tone"
prompt_text = prepare_prompt(lyrics, voice_description)

In [14]:
generate_vocal_audio(models, audio_tokenizer, audio_tokenizer, prompt_text)

TypeError: _forward_unimplemented() got an unexpected keyword argument 'return_tensors'